In [ ]:
# Joahannes B D da Costa <joahannes.costa@unifesp.br>

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.ticker import MultipleLocator

In [ ]:
# cria o diretório para salvar os gráficos, se não existir
dir = "geral"
if not os.path.isdir(dir):
    os.makedirs(dir, exist_ok=True)
    print(f"O diretório '{dir}' foi criado para salvar os gráficos.")
else:
    print(f"O diretório '{dir}' já existe. Os gráficos serão salvos nele.")

In [ ]:
LISTA = "FCFS LOA MAB TEMIS ORION"
LISTA = LISTA.split(" ")

In [ ]:
path = '../system/output/'

output_path = "geral/"

output_name = "Geral"

if not os.path.isdir(output_path):
	os.makedirs(output_path)
else:
	pass

ALGORITHM   = LISTA

WITH_HATCHES = True

HATCHES = {
    'FCFS'  : '..',
	'LOA'   : '\\',
	'MAB'   : 'o',
	'TEMIS' : '////',
	'ORION' : '',
}

LABELS = {
    "FCFS"  : "FCFS",
	"LOA"   : "LOA",
	"MAB"   : "MAB",
	"TEMIS" : "TEMIS",
	"ORION" : "ORION",
}

SUBLEGENDA = {
    '0' : 'a)',
    '1' : 'b)',
    '2' : 'c)',
    '3' : 'd)'
}

X_LEGEND            = u'Prazo (s)'

Y_LEGEND_SCHEDULED  = u'Tarefas Escalonadas (%)'
Y_LEGEND_LATENCY    = u'Latência do Sistema (s)'
Y_LEGEND_COST       = u'Custo Monetário ($)'
Y_LEGEND_CPU_TIME   = u'Tempo de CPU (ms)'

SHAREY = True

header = [
    "task_total","task_id", "task_size", "task_value", "task_cpu", "task_deadline", "task_insert_time", "task_start_time", "task_finish_time", "task_remove_time", "task_waiting_time", "task_cost", "task_status",
    "config_seed", "config_rate", "config_deadline", "algorithm"
    ]

YLIM_SCHEDULED  = (-2, 110)
YLIM_LATENCY    = (-0.15, 6.5)
YLIM_COST       = (-2, 110)
YLIM_CPU_TIME   = (-3, 165)

YTICKS_SCHEDULED = 20
YTICKS_LATENCY   = 1
YTICKS_COST      = 20
YTICKS_CPU_TIME  = 25

In [ ]:
RATE        = [5, 15, 30]  # [1, 2, 3, 4, 5, 10, 15]
DEADLINES   = [0.5, 1, 5, 7] # [0.3, 0.8, 1, 3, 5, 7]
SEEDS       = [1, 2, 3, 4, 5] # [1, 2, 3, 4, 5]
CPU_CYCLE   = 30

In [ ]:
df = pd.DataFrame(columns=header)

dataframes = []

for algorithm in ALGORITHM:

    for task_rate in RATE:

        for deadline in DEADLINES:

            for seed in SEEDS:
    
                arquivo = path + str(algorithm) + '/SEED_' + str(seed) + '_RESULTS_radius_2000_resource_1_weight_10_rate_' + str(task_rate) + '_megacycles_' + str(CPU_CYCLE) + '_deadline_' + str(deadline) + '.txt'
                df = pd.read_csv(arquivo, sep='\t', names=header)
                df['config_seed'] = seed
                df['config_rate'] = task_rate
                df['config_deadline'] = deadline
                df['algorithm'] = algorithm
                dataframes.append(df)

                # print(arquivo)

df_final = pd.concat(dataframes, ignore_index=True)
            

## Criando coluna para Latency

In [ ]:
# em segundos
df_final['result_latency'] = (df_final['task_remove_time'] - df_final['task_insert_time'])
# df_final['result_latency'] = df_final['result_latency'] / 1000
df_final

## Configuração dos plots

In [ ]:
formato = ".png"

legend_size = 15

x_fig = 12 #6.8
y_fig = 4 #5.5

plot_style = "ticks"
# plot_palette = "tab10"
# plot_palette = "Greys"
# plot_palette = "tab20c"
# color_style = "Blues"

plot_palette = ['#6D6D6D', '#929292', '#B6B6B6', '#DBDBDB', '#E6550D']

font_size_algoritm = 17

config_legend = (0.5, 1.15)

grid_config_lw    = 1.6
grid_config_alpha = 0.05

plot_title = r" Taxa de chegada $\lambda$ = "

# Scheduled Tasks

In [ ]:


def plot_scheduled_juntos(RATE, ALGORITHM):

    fig, ax = plt.subplots(nrows=1, ncols=len(RATE), figsize=(x_fig,y_fig), sharey=SHAREY, layout='constrained')

    for rate in range(len(RATE)):

        df_local = df_final[df_final['config_rate'] == RATE[rate]]
        
        sns.set_theme(style = plot_style, palette = plot_palette)
    
        lol = sns.barplot(
            ax=ax[rate],
            x='config_deadline',
            y=(df_local['task_status'] == 'COMPLETED') * 100,
            data=df_local,
            hue='algorithm',
            edgecolor='k',
            estimator=np.mean,
            errorbar=('ci', 95),
            capsize=.08,
            err_kws={'linewidth': 1},
        )

        if WITH_HATCHES == True:
            hatches = HATCHES.values()
            # Loop over the bars
            for bars, hatch in zip(lol.containers, hatches):
                # Set a different hatch for each group of bars
                for bar in bars:
                    bar.set_hatch(hatch)

        # Remove legend do subplot
        ax[rate].get_legend().remove()

        ax[rate].set_axisbelow(True)

        ax[rate].set_ylim(YLIM_SCHEDULED)

        # Define labels de X e Y
        ax[rate].set_xlabel(X_LEGEND, fontsize=legend_size)
        ax[rate].set_ylabel(Y_LEGEND_SCHEDULED, fontsize=legend_size)

        # Define title para cada subplot
        ax[rate].set_title(str(SUBLEGENDA[str(rate)]) + plot_title + str(RATE[rate]), fontsize=legend_size)

        # Configuração do tamanho da fonte para xticks e yticks
        ax[rate].tick_params(axis='x', labelsize=legend_size)  # Ajuste o valor de labelsize conforme necessário
        ax[rate].tick_params(axis='y', labelsize=legend_size)  # Ajuste o valor de labelsize conforme necessário

        # Define os intervalos dos ticks do eixo y para cada subplot
        ax[rate].yaxis.set_major_locator(MultipleLocator(YTICKS_SCHEDULED))

        # Define formato do grid
        ax[rate].grid(color='k', linestyle='--', linewidth=grid_config_lw, axis='both', alpha=grid_config_alpha)
    
    # Adiciona um bloco de legenda no topo dos subplots com hatches
    legend_handles, names = ax[0].get_legend_handles_labels()
    final_labels = []
    if WITH_HATCHES == True:
        for alg, handle, hatch in zip(names, legend_handles, HATCHES):
            final_name = LABELS[alg]
            final_labels.append(final_name)
            handle.set_hatch(HATCHES[alg])
    else:
        for alg, handle in zip(names, legend_handles):
            final_name = LABELS[alg]
            final_labels.append(final_name)

    fig.legend(
        legend_handles,
        final_labels,
        loc             = 'upper center',
        bbox_to_anchor  = config_legend,
        # handletextpad   = 0.2,
        handlelength    = 2.2,
        handleheight    = 1.2,
        columnspacing   = 1,
        fancybox        = False,
        frameon         = False,
        ncol            = len(ALGORITHM),
        edgecolor       = 'k',
        fontsize        = font_size_algoritm
        )

    fig = plt.gcf()
    
    # fig.tight_layout()

    # fig.set_size_inches(x_fig, y_fig)
    fig.savefig(output_path + 'Escalonadas_' + output_name + formato, dpi=200, bbox_inches = 'tight', pad_inches = 0.05)

    plt.show()

# System Latency

In [ ]:

def plot_latency_juntos(RATE, ALGORITHM):

    fig, ax = plt.subplots(nrows=1, ncols=len(RATE), figsize=(x_fig,y_fig), sharey=SHAREY, layout='constrained')

    for rate in range(len(RATE)):

        df_local = df_final[df_final['config_rate'] == RATE[rate]]
        # df_local = df_local[df_local['task_status'] == "COMPLETED"]

        sns.set_theme(style = plot_style, palette = plot_palette)
    
        lol = sns.barplot(
            ax=ax[rate],
            x='config_deadline',
            y='result_latency',
            data=df_local,
            hue='algorithm',
            edgecolor='k',
            estimator=np.mean,
            errorbar=('ci', 95),
            capsize=.08,
            err_kws={'linewidth': 1},
        )
        
        if WITH_HATCHES == True:
            hatches = HATCHES.values()
            # Loop over the bars
            for bars, hatch in zip(lol.containers, hatches):
                # Set a different hatch for each group of bars
                for bar in bars:
                    bar.set_hatch(hatch)

        # Remove legend do subplot
        ax[rate].get_legend().remove()

        ax[rate].set_axisbelow(True)

        ax[rate].set_ylim(YLIM_LATENCY)

        # Define labels de X e Y
        ax[rate].set_xlabel(X_LEGEND, fontsize=legend_size)
        ax[rate].set_ylabel(Y_LEGEND_LATENCY, fontsize=legend_size)

        # Define title para cada subplot
        ax[rate].set_title(str(SUBLEGENDA[str(rate)])  + plot_title + str(RATE[rate]), fontsize=legend_size)

        # Configuração do tamanho da fonte para xticks e yticks
        ax[rate].tick_params(axis='x', labelsize=legend_size)  # Ajuste o valor de labelsize conforme necessário
        ax[rate].tick_params(axis='y', labelsize=legend_size)  # Ajuste o valor de labelsize conforme necessário

        # Define os intervalos dos ticks do eixo y para cada subplot
        ax[rate].yaxis.set_major_locator(MultipleLocator(YTICKS_LATENCY))

        # Define formato do grid
        ax[rate].grid(color='k', linestyle='--', linewidth=grid_config_lw, axis='both', alpha=grid_config_alpha)
    
    # Adiciona um bloco de legenda no topo dos subplots com hatches
    legend_handles, names = ax[0].get_legend_handles_labels()
    final_labels = []
    if WITH_HATCHES == True:
        for alg, handle, hatch in zip(names, legend_handles, HATCHES):
            final_name = LABELS[alg]
            final_labels.append(final_name)
            handle.set_hatch(HATCHES[alg])
    else:
        for alg, handle in zip(names, legend_handles):
            final_name = LABELS[alg]
            final_labels.append(final_name)

    fig.legend(
        legend_handles,
        final_labels,
        loc             = 'upper center',
        bbox_to_anchor  = config_legend,
        # handletextpad   = 0.2,
        handlelength    = 2.2,
        handleheight    = 1.2,
        columnspacing   = 1,
        fancybox        = False,
        frameon         = False,
        ncol            = len(ALGORITHM),
        edgecolor       = 'k',
        fontsize        = font_size_algoritm
        )

    fig = plt.gcf()
    # fig.set_size_inches(x_fig, y_fig)
    fig.savefig(output_path + 'Latencia_' + output_name + formato, dpi=200, bbox_inches = 'tight', pad_inches = 0.05)

    plt.show()


# Monetary Costs

In [ ]:
def plot_cost_juntos(RATE, ALGORITHM):

    fig, ax = plt.subplots(nrows=1, ncols=len(RATE), figsize=(x_fig,y_fig), sharey=SHAREY, layout='constrained')

    for rate in range(len(RATE)):

        df_local = df_final[df_final['config_rate'] == RATE[rate]]
        # df_local = df_local[df_local['task_status'] == "COMPLETED"]

        sns.set_theme(style = plot_style, palette = plot_palette)
    
        lol = sns.barplot(
            ax=ax[rate],
            x='config_deadline',
            y='task_cost',
            data=df_local,
            hue='algorithm',
            edgecolor='k',
            estimator=np.mean,
            errorbar=('ci', 95),
            capsize=.08,
            err_kws={'linewidth': 1},
        )
        
        if WITH_HATCHES == True:
            hatches = HATCHES.values()
            # Loop over the bars
            for bars, hatch in zip(lol.containers, hatches):
                # Set a different hatch for each group of bars
                for bar in bars:
                    bar.set_hatch(hatch)

        # Remove legend do subplot
        ax[rate].get_legend().remove()

        ax[rate].set_axisbelow(True)

        ax[rate].set_ylim(YLIM_COST)

        # Define labels de X e Y
        ax[rate].set_xlabel(X_LEGEND, fontsize=legend_size)
        ax[rate].set_ylabel(Y_LEGEND_COST, fontsize=legend_size)

        # Define title para cada subplot
        ax[rate].set_title(str(SUBLEGENDA[str(rate)]) + plot_title + str(RATE[rate]), fontsize=legend_size)

        # Configuração do tamanho da fonte para xticks e yticks
        ax[rate].tick_params(axis='x', labelsize=legend_size)  # Ajuste o valor de labelsize conforme necessário
        ax[rate].tick_params(axis='y', labelsize=legend_size)  # Ajuste o valor de labelsize conforme necessário

        # Define os intervalos dos ticks do eixo y para cada subplot
        ax[rate].yaxis.set_major_locator(MultipleLocator(YTICKS_COST))

        # Define formato do grid
        ax[rate].grid(color='k', linestyle='--', linewidth=grid_config_lw, axis='both', alpha=grid_config_alpha)
    
    # Adiciona um bloco de legenda no topo dos subplots com hatches
    legend_handles, names = ax[0].get_legend_handles_labels()
    final_labels = []
    if WITH_HATCHES == True:
        for alg, handle, hatch in zip(names, legend_handles, HATCHES):
            final_name = LABELS[alg]
            final_labels.append(final_name)
            handle.set_hatch(HATCHES[alg])
    else:
        for alg, handle in zip(names, legend_handles):
            final_name = LABELS[alg]
            final_labels.append(final_name)

    # hatches
    hatches = HATCHES.values()
    for hatch in hatches:   
        handle.set_hatch(hatch)

    fig.legend(
        legend_handles,
        final_labels,
        loc             = 'upper center',
        bbox_to_anchor  = config_legend,
        # handletextpad   = 0.2,
        handlelength    = 2.2,
        handleheight    = 1.2,
        columnspacing   = 1,
        fancybox        = False,
        frameon         = False,
        ncol            = len(ALGORITHM),
        edgecolor       = 'k',
        fontsize        = font_size_algoritm
        )

    fig = plt.gcf()
    # fig.set_size_inches(x_fig, y_fig)
    fig.savefig(output_path + 'Custo_' + output_name + formato, dpi=200, bbox_inches = 'tight', pad_inches = 0.05)

    plt.show()

In [ ]:
plot_scheduled_juntos(RATE, ALGORITHM)
plot_latency_juntos(RATE, ALGORITHM)
plot_cost_juntos(RATE, ALGORITHM)

# CPU Time

In [ ]:
header = ['cpu_time', 'config_seed', 'config_rate', 'config_deadline', 'algorithm']

df = pd.DataFrame(columns=header)

dataframes = []

for algorithm in ALGORITHM:

    for task_rate in RATE:

        for deadline in DEADLINES:

            for seed in SEEDS:
    
                arquivo = path + str(algorithm) + '/SEED_' + str(seed) + '_TIME_radius_2000_resource_1_weight_10_rate_' + str(task_rate) + '_megacycles_' + str(CPU_CYCLE) + '_deadline_' + str(deadline) + '.txt'
                df = pd.read_csv(arquivo, sep='\t', names=header)
                df['config_seed'] = seed
                df['config_rate'] = task_rate
                df['config_deadline'] = deadline
                df['algorithm'] = algorithm
                dataframes.append(df)

                # print(arquivo)

df_final = pd.concat(dataframes, ignore_index=True)

In [ ]:
df_final['cpu_time_ms'] = df_final['cpu_time'] * 1000
df_final

# Execução CPU Time

In [ ]:


def plot_cputime_juntos(RATE, ALGORITHM):

    fig, ax = plt.subplots(nrows=1, ncols=len(RATE), figsize=(x_fig,y_fig), sharey=True, layout='constrained')

    for rate in range(len(RATE)):

        df_time = df_final[df_final['config_rate'] == RATE[rate]]

        sns.set_theme(style = plot_style, palette = plot_palette)
    
        lol = sns.barplot(
            ax=ax[rate],
            x='config_deadline',
            y='cpu_time_ms',
            data=df_time,
            hue='algorithm',
            edgecolor='k',
            estimator=np.mean,
            errorbar=('ci', 95),
            capsize=.08,
            err_kws={'linewidth': 1},
        )
        
        if WITH_HATCHES == True:
            hatches = HATCHES.values()
            # Loop over the bars
            for bars, hatch in zip(lol.containers, hatches):
                # Set a different hatch for each group of bars
                for bar in bars:
                    bar.set_hatch(hatch)

        # Remove legend do subplot
        ax[rate].get_legend().remove()

        ax[rate].set_axisbelow(True)

        ax[rate].set_ylim(YLIM_CPU_TIME)

        # ax[rate].set_yscale('log')

        # Define labels de X e Y
        ax[rate].set_xlabel(X_LEGEND, fontsize=legend_size)
        ax[rate].set_ylabel(Y_LEGEND_CPU_TIME, fontsize=legend_size)

        # Define title para cada subplot
        ax[rate].set_title(str(SUBLEGENDA[str(rate)]) + plot_title + str(RATE[rate]), fontsize=legend_size)

        # Configuração do tamanho da fonte para xticks e yticks
        ax[rate].tick_params(axis='x', labelsize=legend_size)  # Ajuste o valor de labelsize conforme necessário
        ax[rate].tick_params(axis='y', labelsize=legend_size)  # Ajuste o valor de labelsize conforme necessário

        # Define os intervalos dos ticks do eixo y para cada subplot
        ax[rate].yaxis.set_major_locator(MultipleLocator(YTICKS_CPU_TIME))

        # Define formato do grid
        ax[rate].grid(color='k', linestyle='--', linewidth=grid_config_lw, axis='both', alpha=grid_config_alpha)
    
    # Adiciona um bloco de legenda no topo dos subplots com hatches
    legend_handles, names = ax[0].get_legend_handles_labels()
    final_labels = []
    if WITH_HATCHES == True:
        for alg, handle, hatch in zip(names, legend_handles, HATCHES):
            final_name = LABELS[alg]
            final_labels.append(final_name)
            handle.set_hatch(HATCHES[alg])
    else:
        for alg, handle in zip(names, legend_handles):
            final_name = LABELS[alg]
            final_labels.append(final_name)

    fig.legend(
        legend_handles,
        final_labels,
        loc             = 'upper center',
        bbox_to_anchor  = config_legend,
        # handletextpad   = 0.2,
        handlelength    = 2.2,
        handleheight    = 1.2,
        columnspacing   = 1,
        fancybox        = False,
        frameon         = False,
        ncol            = len(ALGORITHM),
        edgecolor       = 'k',
        fontsize        = font_size_algoritm
        )

    fig = plt.gcf()
    # fig.set_size_inches(x_fig, y_fig)
    fig.savefig(output_path + 'TempoCPU_' + output_name + formato, dpi=200, bbox_inches = 'tight', pad_inches = 0.05)

    plt.show()

In [ ]:
plot_cputime_juntos(RATE, ALGORITHM)